In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [3]:
df = pd.read_csv('train.csv')

In [4]:
new_df = df.sample(30000)

In [5]:
new_df.isnull().sum()

id              0
qid1            0
qid2            0
question1       0
question2       1
is_duplicate    0
dtype: int64

In [7]:
new_df.duplicated().sum()

np.int64(0)

In [8]:
new_df.head()

,id,qid1,qid2,question1,question2,is_duplicate
354527,354527,346089,483662,"Who is stronger, Luke Cage or Jessica Jones?",Who will win in a bed between Luke Cage and Sh...,0
117785,117785,143192,191587,"How many cards are in a deck? Also, how many o...",Two cards are drawn without replacement from a...,0
105209,105209,173550,173551,What is the cutoff for AAI Junior Executive (c...,what is the syllabus for AAI junior Executive ...,1
348135,348135,476666,138982,Oklahoma State Football Live Stream | Watch Ok...,Indiana Football Live Stream | Watch Indiana H...,0
341945,341945,469853,54009,Am I meditating properly?,How do I meditate properly?,0


In [9]:
ques_df = new_df[['question1', 'question2']]

In [10]:
ques_df.head()

,question1,question2
354527,"Who is stronger, Luke Cage or Jessica Jones?",Who will win in a bed between Luke Cage and Sh...
117785,"How many cards are in a deck? Also, how many o...",Two cards are drawn without replacement from a...
105209,What is the cutoff for AAI Junior Executive (c...,what is the syllabus for AAI junior Executive ...
348135,Oklahoma State Football Live Stream | Watch Ok...,Indiana Football Live Stream | Watch Indiana H...
341945,Am I meditating properly?,How do I meditate properly?


In [14]:
ques_df['question1'] = ques_df['question1'].fillna('')
ques_df['question2'] = ques_df['question2'].fillna('')

/var/folders/s7/p1d6kl217wn7_y52f1sn0d580000gn/T/ipykernel_32027/3551104273.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ques_df['question1'] = ques_df['question1'].fillna('')
/var/folders/s7/p1d6kl217wn7_y52f1sn0d580000gn/T/ipykernel_32027/3551104273.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ques_df['question2'] = ques_df['question2'].fillna('')


In [15]:
from sklearn.feature_extraction.text import CountVectorizer

questions = list(ques_df['question1']) + list(ques_df['question2'])
cv = CountVectorizer(max_features=3000)
q1_arr1, q2_arr2 = np.vsplit(cv.fit_transform(questions).toarray(), 2)

In [16]:
temp_df1 = pd.DataFrame(q1_arr1, index = ques_df.index)
temp_df2 = pd.DataFrame(q2_arr2, index = ques_df.index)
temp_df = pd.concat([temp_df2, temp_df2], axis = 1)
temp_df.shape

(30000, 6000)

In [17]:
temp_df['is_duplicate'] = new_df['is_duplicate']

In [18]:
temp_df.head()

,0,1,2,3,4,5,6,7,8,9,...,2991,2992,2993,2994,2995,2996,2997,2998,2999,is_duplicate
354527,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
117785,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
105209,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
348135,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
341945,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [21]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(temp_df.iloc[:,0:-1].values, temp_df.iloc[:,-1].values, test_size = 0.2, random_state = 42)

In [22]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
rf = RandomForestClassifier()
rf.fit(x_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [23]:
y_pred = rf.predict(x_test)
accuracy_score(y_test, y_pred)

0.7168333333333333

In [25]:
!pip install xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 11.2 MB/s eta 0:00:0031m11.8 MB/s eta 0:00:01


In [26]:
from xgboost import XGBClassifier
xgb = XGBClassifier()
xgb.fit(x_train, y_train)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [27]:
y_pred1 = xgb.predict(x_test)
accuracy_score(y_test, y_pred)

0.7168333333333333